# Visual-language temporal boundary alignment
Run from the top in a GPU Colab runtime. Add a `GITHUB_TOKEN` Colab secret with read access to the private repository.

In [ ]:
# Clone or update the private repository, then install dependencies.
from pathlib import Path
import base64, getpass, os, subprocess, sys

REPO = 'sanay-tralshawala/video-language-temporal-boundaries'
REPO_DIR = Path('/content/video-language-temporal-boundaries')
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('GitHub token with read access to the private repository: ')
if not token:
    raise RuntimeError('GITHUB_TOKEN is required because this repository is private.')
credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_auth = ['git', '-c', f'http.extraHeader=AUTHORIZATION: basic {credential}']
if (REPO_DIR / '.git').is_dir():
    result = subprocess.run(git_auth + ['-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    result = subprocess.run(git_auth + ['clone', '--depth', '1', f'https://github.com/{REPO}.git', str(REPO_DIR)])
token = credential = None
if result.returncode:
    raise RuntimeError('Repository clone/update failed; check GITHUB_TOKEN access.')
os.chdir(REPO_DIR)
requirements = REPO_DIR / 'requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Repository ready:', REPO_DIR)

In [ ]:
# Mount persistent output storage and report the accelerator.
from google.colab import drive
drive.mount('/content/drive')
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Choose the study mode
Keep `two_video_trial` for a quick check. Use `stratified_24` for 24 reproducibly sampled long videos with at least three from each domain and maximum subcategory coverage. The full mode cycles through official archives, extracts selected files, and deletes each ZIP.

In [ ]:
STUDY_MODE = 'stratified_24'  # use 'two_video_trial' for a quick smoke test
OUTPUT_DIR = '/content/drive/MyDrive/video_boundary_results'
DATASET_DIR_24 = '/content/Video-MME-stratified-24'
SAMPLE_SIZE = 24
MIN_PER_DOMAIN = 3
RANDOM_SEED = 0
FRAME_INTERVAL = 2.0
SUBTITLE_WINDOW = 15.0

In [ ]:
# Prepare either the two-video trial or the fixed, committed 24-video selection.
from pathlib import Path
import pandas as pd

VIDEO_LIST = None
METADATA_CSV = None
if STUDY_MODE == 'two_video_trial':
    from download_trial_subset import download_two_video_trial
    DATASET_DIR = str(download_two_video_trial('/content/Video-MME-two-video-trial'))
elif STUDY_MODE == 'stratified_24':
    from download_selected_subset import download_selected_subset
    from sample_videos import build_pipeline_manifest
    selection_csv = REPO_DIR / 'manifests' / 'long_stratified_24.csv'
    if not selection_csv.is_file():
        raise FileNotFoundError(f'Committed selection manifest not found: {selection_csv}')
    selected = pd.read_csv(selection_csv, dtype={'video_id': str})
    if len(selected) != SAMPLE_SIZE:
        raise RuntimeError(f'Expected {SAMPLE_SIZE} rows in {selection_csv}, found {len(selected)}')
    display(selected)
    DATASET_DIR = str(download_selected_subset(selection_csv, DATASET_DIR_24))
    pipeline_manifest = Path(OUTPUT_DIR) / 'manifests' / 'pipeline_manifest.csv'
    METADATA_CSV = str(build_pipeline_manifest(
        selection_csv, DATASET_DIR, pipeline_manifest))
    print('Using committed video list:', selection_csv)
    print('Domain counts:', selected['domain'].value_counts().to_dict())
    print('Covered subcategories:', selected[['domain', 'sub_category']].drop_duplicates().shape[0])
else:
    raise ValueError("STUDY_MODE must be 'two_video_trial' or 'stratified_24'")

In [ ]:
# Validate every selected video/subtitle pair before loading models.
from video_boundary_pipeline import discover_videos
pairs = discover_videos(DATASET_DIR, VIDEO_LIST, METADATA_CSV, max_videos=SAMPLE_SIZE)
expected = 2 if STUDY_MODE == 'two_video_trial' else SAMPLE_SIZE
if len(pairs) != expected:
    raise RuntimeError(f'Expected {expected} valid pairs, found {len(pairs)}')
print(f'Validated {len(pairs)} video/subtitle pairs')
for pair in pairs[:5]: print(' ', pair.video_id, pair.video_path.name, '+', pair.subtitle_path.name)

In [ ]:
# Run the cached analysis and save change plus stacked-similarity figures.
from video_boundary_pipeline import PipelineConfig, run_pipeline
config = PipelineConfig(dataset_dir=DATASET_DIR, output_dir=OUTPUT_DIR,
    frame_interval=FRAME_INTERVAL, subtitle_window=SUBTITLE_WINDOW,
    max_videos=expected)
results = run_pipeline(config, video_list=VIDEO_LIST, metadata_csv=METADATA_CSV)
display(results['alignment'])
display(results['stats'])

## Peak table and graphs
The table contains every detected peak, its strength, nearest cross-modal distance, alignment type, and nearby subtitle text. Figures are loaded from the persisted output directory.

In [ ]:
# Display and link the complete peak-event table.
from IPython.display import FileLink, Image, Markdown, display
peak_table = results['events'].sort_values(
    ['video_id', 'timestamp_seconds', 'modality']).reset_index(drop=True)
display(peak_table)
peak_csv = Path(OUTPUT_DIR) / 'aggregate' / 'all_events.csv'
display(FileLink(str(peak_csv), result_html_prefix='Download the full peak table: '))

In [ ]:
# Display the change/peak and stacked-similarity graphs for every analyzed video.
figure_dir = Path(OUTPUT_DIR) / 'figures'
for video_id in results['stats']['video_id'].astype(str):
    display(Markdown(f'### Video {video_id}'))
    for suffix in ('.png', '_similarities.png'):
        image_path = figure_dir / f'{video_id}{suffix}'
        if image_path.is_file():
            display(Image(filename=str(image_path), width=1100))

display(Markdown('## Aggregate graphs'))
for image_path in sorted((Path(OUTPUT_DIR) / 'aggregate').glob('*.png')):
    title = image_path.stem.replace('_', ' ').title()
    display(Markdown(f'### {title}'))
    display(Image(filename=str(image_path), width=900))